UED-Workflow in Schweppes Group in Hamburg
Code Author Helena Hollstein

Fill out the next cell, save and then run cells individually but in the correct order!

In [ ]:
import sys
import pandas as pd
from pathlib import Path
from uedhhlib.datasets import PumpedDataset
from uedhhlib.io.writers import save_datasets_h5
from uedhhlib.tools import DataPicker
from pyqtgraph.Qt import QtWidgets

meas_path = Path(r"Z:\Users\Emma\Zyla1\2026\2026_11\NT52\261117\Au\Meas3") # path until MeasX
material_string = "Au"
exp_nr = 16 # Number of UED experiments on this material
cycles = (1, 3) #either as list, int or tuple(then from first to last entry including first and last entry)

In [ ]:
#Load the data as hdf5 file for the first time to get the raw dataset

pumped = PumpedDataset(meas_path,
                       progress=True,
                       mask_dir = None,
                       cycles = cycles,
                       ignored_labels = None,
                       ignored_jsonpath = None
                       )
pumped.process(correct_dark=False,
               correct_laser=False,
               output_dir = None,
               unpumped_fname = None)
save_datasets_h5(pumped, 
             Path(meas_path / "hlh_analysis" /f"exp{exp_nr}_raw.h5"))

In [ ]:
# Use the datapicker to exclude sparcs. When the window opens, add aread with "+" and at the end save by "ctrl"+"S" as json file.
# CAUTION: If you give the jscon file a different name than "ignored_labels.json", you must change the "ignored_jsonpath" name in the next cell.

raw_h5_fpath = r"Z:\Users\Emma\Zyla1\2026\2026_11\NT52\261117\Au\Meas3\hlh_analysis\exp16_raw.h5"

df = pd.read_hdf(raw_h5_fpath, key="file_registry")

app = QtWidgets.QApplication(sys.argv)
win = DataPicker(df, default_dir=str(Path(raw_h5_fpath).parent))
sys.exit(app.exec())

In [ ]:
#Load the data as hdf5 file for the second time to get the dataset without arcs and corrected for dark and laser background.
# You also get a mean unpumped image as .npy file (use this one for pyfai-calib2)

pumped = PumpedDataset(meas_path,
                       progress=True,
                       mask_dir = None,
                       cycles = cycles,
                       ignored_labels = None,
                       ignored_jsonpath = meas_path /"hlh_analysis"/"ignored_labels.json"
                       )
pumped.process(correct_dark=True,
               correct_laser=True,
               output_dir = None, #measX_path/"hlh_analysis",
               unpumped_fname = None) #"exp6_unpumped_woarcs.h5")
save_datasets_h5(pumped, 
             meas_path/"hlh_analysis"/f"exp{exp_nr}_woarcs.h5")